# DTM: Keyword Trend Evaluation (Scenario 4)

1. **Ground Truth**: TF-IDF per year → classify keywords as Emerging / Stable / Decaying
2. **SPAN**: Longest consecutive years each keyword appears in model topics

In [1]:
import pandas as pd
import numpy as np
import ast
from pathlib import Path
from sklearn.feature_extraction.text import TfidfVectorizer
from collections import defaultdict
import warnings
warnings.filterwarnings("ignore")

In [2]:
LIST_SUBJECT = ["cs", "math", "physics"]
DATA_DIR = Path("../../../../data/preprocess")
TEMPORAL_DIR = Path("../../../../results/dtm/temporal")
RESULT_DIR = Path("../../../../results/dtm/tren")

TOP_K = 20  # top keywords per category
EARLY_YEARS = range(2000, 2011)   # 2000-2010
LATE_YEARS = range(2015, 2026)     # 2015-2025

for subject in LIST_SUBJECT:
    (RESULT_DIR / subject).mkdir(parents=True, exist_ok=True)

print(f"Data: {DATA_DIR}")
print(f"Topics: {TEMPORAL_DIR}")
print(f"Output: {RESULT_DIR}")
print(f"Top-K keywords per category: {TOP_K}")

Data: ../../../../data/preprocess
Topics: ../../../../results/dtm/temporal
Output: ../../../../results/dtm/tren
Top-K keywords per category: 20


## Step 1: Compute TF-IDF per Year
Calculate average TF-IDF score for each word in each year.

In [3]:
def compute_yearly_tfidf(subject):
    df = pd.read_csv(DATA_DIR / subject / "bow/v1.csv")
    df["year"] = pd.to_datetime(df["submitted_date"]).dt.year

    # Parse text: could be string repr of list or plain text
    def to_text(val):
        try:
            tokens = ast.literal_eval(val)
            if isinstance(tokens, list):
                return " ".join(tokens)
        except (ValueError, SyntaxError):
            pass
        return str(val)

    df["text_str"] = df["text"].apply(to_text)
    years = sorted(df["year"].unique())

    # Compute TF-IDF per year
    yearly_scores = {}  # word → {year: avg_tfidf}
    for year in years:
        year_docs = df[df["year"] == year]["text_str"].tolist()
        if len(year_docs) < 5:
            continue

        tfidf = TfidfVectorizer(max_features=5000, min_df=2)
        matrix = tfidf.fit_transform(year_docs)
        feature_names = tfidf.get_feature_names_out()
        avg_scores = np.asarray(matrix.mean(axis=0)).flatten()

        for word, score in zip(feature_names, avg_scores):
            if word not in yearly_scores:
                yearly_scores[word] = {}
            yearly_scores[word][year] = float(score)

    return yearly_scores, years

# Cache results
tfidf_cache = {}
for subject in LIST_SUBJECT:
    print(f"Computing TF-IDF for {subject}...")
    tfidf_cache[subject] = compute_yearly_tfidf(subject)
    n_words = len(tfidf_cache[subject][0])
    print(f"  {n_words} unique words tracked")

Computing TF-IDF for cs...
  12665 unique words tracked
Computing TF-IDF for math...
  13547 unique words tracked
Computing TF-IDF for physics...
  13106 unique words tracked


## Step 2: Classify Keywords
- **Emerging**: low early (2000-2010), high late (2015-2025)
- **Stable**: consistently high across all years
- **Decaying**: high early, low late

In [4]:
def classify_keywords(yearly_scores, years, top_k=20):
    """
    Classify words using LINEAR REGRESSION SLOPE of TF-IDF over time.
    - Emerging: strong positive slope (growing importance)
    - Stable: near-zero slope + high avg TF-IDF (consistently important)
    - Decaying: strong negative slope (declining importance)
    """
    from scipy.stats import linregress

    word_stats = []
    years_arr = np.array(years, dtype=float)

    for word, scores in yearly_scores.items():
        # Get TF-IDF values aligned with years
        vals = np.array([scores.get(y, 0.0) for y in years])

        # Only consider words present in at least 3 years
        n_present = np.sum(vals > 0)
        if n_present < 3:
            continue

        # Linear regression: TF-IDF = a + slope * year
        slope, intercept, r_val, p_val, std_err = linregress(years_arr, vals)

        overall_avg = vals.mean()
        overall_std = vals.std()

        # First and last year the word appears
        present_years = [y for y, v in zip(years, vals) if v > 0]
        first_year = min(present_years)
        last_year = max(present_years)

        # Early vs late avg (for display)
        early = [y for y in years if y <= 2010]
        late = [y for y in years if y >= 2015]
        early_avg = np.mean([scores.get(y, 0.0) for y in early])
        late_avg = np.mean([scores.get(y, 0.0) for y in late])

        word_stats.append({
            'word': word,
            'slope': slope,
            'r_squared': r_val**2,
            'p_value': p_val,
            'overall_avg': overall_avg,
            'overall_std': overall_std,
            'early_avg': early_avg,
            'late_avg': late_avg,
            'n_present': n_present,
            'first_year': first_year,
            'last_year': last_year,
        })

    stats_df = pd.DataFrame(word_stats)

    # Emerging: highest positive slope (statistically significant)
    emerging_pool = stats_df[stats_df['slope'] > 0]
    emerging = emerging_pool.nlargest(top_k, 'slope')

    # Stable: high avg TF-IDF + lowest absolute slope → consistent importance
    # Score: avg / (1 + |slope| * 10000)  → rewards high avg, penalizes any trend
    used = set(emerging['word'])
    stable_pool = stats_df[~stats_df['word'].isin(used)].copy()
    stable_pool['stability'] = stable_pool['overall_avg'] / (1 + stable_pool['slope'].abs() * 10000)
    stable = stable_pool.nlargest(top_k, 'stability')

    # Decaying: strongest negative slope
    used.update(stable['word'])
    decay_pool = stats_df[(~stats_df['word'].isin(used)) & (stats_df['slope'] < 0)]
    decaying = decay_pool.nsmallest(top_k, 'slope')

    return emerging, stable, decaying, stats_df

# Classify and save
keyword_cache = {}
for subject in LIST_SUBJECT:
    print(f"\n{'='*70}")
    print(f"Keyword Classification: {subject.upper()}")
    print(f"{'='*70}")

    yearly_scores, years = tfidf_cache[subject]
    emerging, stable, decaying, stats_df = classify_keywords(yearly_scores, years, TOP_K)
    keyword_cache[subject] = (emerging, stable, decaying)

    # Save ground truth
    gt_rows = []
    for cat, cat_df in [('emerging', emerging), ('stable', stable), ('decaying', decaying)]:
        for _, r in cat_df.iterrows():
            gt_rows.append({
                'word': r['word'], 'category': cat,
                'slope': round(r['slope'], 8),
                'r_squared': round(r['r_squared'], 4),
                'early_avg': round(r['early_avg'], 6),
                'late_avg': round(r['late_avg'], 6),
                'overall_avg': round(r['overall_avg'], 6),
                'first_year': int(r['first_year']),
                'n_present': int(r['n_present']),
            })
    gt_df = pd.DataFrame(gt_rows)
    gt_df.to_csv(RESULT_DIR / subject / 'ground_truth_keywords.csv', index=False)

    for cat, icon, cat_df in [
        ('Emerging', chr(0x1F4C8), emerging),
        ('Stable', chr(0x1F512), stable),
        ('Decaying', chr(0x1F4C9), decaying),
    ]:
        print(f"\n  {icon} {cat} (top {TOP_K}):")
        print(f"  {'Word':25s} {'Slope':>10s} {'R²':>6s} {'Early':>8s} {'Late':>8s} {'1st yr':>6s}")
        print(f"  {'-'*70}")
        for _, r in cat_df.head(10).iterrows():
            print(f"  {r['word']:25s} {r['slope']:10.6f} {r['r_squared']:6.3f} "
                  f"{r['early_avg']:8.5f} {r['late_avg']:8.5f} {int(r['first_year']):>6d}")

    print(f"\n  Saved: {RESULT_DIR / subject / 'ground_truth_keywords.csv'}")


Keyword Classification: CS

  📈 Emerging (top 20):
  Word                           Slope     R²    Early     Late 1st yr
  ----------------------------------------------------------------------
  image                       0.001015  0.898  0.00648  0.02293   2000
  dataset                     0.000889  0.869  0.00174  0.01587   2000
  training                    0.000642  0.752  0.00265  0.01286   2000
  deep                        0.000509  0.640  0.00055  0.00987   2002
  video                       0.000478  0.921  0.00177  0.00944   2000
  learning                    0.000451  0.772  0.00428  0.01173   2000
  llm                         0.000446  0.278  0.00000  0.00503   2022
  human                       0.000444  0.864  0.00410  0.01067   2000
  datum                       0.000431  0.658  0.01887  0.02620   2000
  deep_learning               0.000398  0.719  0.00000  0.00662   2012

  🔒 Stable (top 20):
  Word                           Slope     R²    Early     Late 1st yr
 

## Step 3: SPAN Calculation
For each ground truth keyword, check presence in model topics per year.
SPAN = longest consecutive year sequence the keyword appears in any topic.

In [5]:
def compute_span(keyword, topic_words_by_year, years):
    """
    Compute SPAN and topic count per year for a keyword.
    Returns: max_span, total_present, presence, best_start, topic_counts
    """
    presence = []
    topic_counts = []  # number of topics containing this word per year
    for y in years:
        count = sum(1 for words in topic_words_by_year.get(y, []) if keyword in words)
        topic_counts.append(count)
        presence.append(1 if count > 0 else 0)

    max_span = 0
    current_span = 0
    span_start = None
    best_start = None
    for i, p in enumerate(presence):
        if p == 1:
            if current_span == 0:
                span_start = years[i]
            current_span += 1
            if current_span > max_span:
                max_span = current_span
                best_start = span_start
        else:
            current_span = 0

    total_present = sum(presence)
    return max_span, total_present, presence, best_start, topic_counts


for subject in LIST_SUBJECT:
    print(f"\n{'='*70}")
    print(f"SPAN Analysis: {subject.upper()}")
    print(f"{'='*70}")

    evo_df = pd.read_csv(TEMPORAL_DIR / subject / "topic_word_evolution.csv")
    years = sorted(evo_df['year'].unique())

    topic_words_by_year = defaultdict(list)
    for _, row in evo_df.iterrows():
        words = set(w.strip() for w in str(row['top_words']).split(','))
        topic_words_by_year[int(row['year'])].append(words)

    emerging, stable, decaying = keyword_cache[subject]

    span_rows = []
    for cat, cat_name, cat_df in [
        ('emerging', chr(0x1F4C8) + ' Emerging', emerging),
        ('stable', chr(0x1F512) + ' Stable', stable),
        ('decaying', chr(0x1F4C9) + ' Decaying', decaying),
    ]:
        print(f"\n  {cat_name}:")
        # Header with year labels
        yr_labels = ''.join([str(y)[-2:] for y in years])
        print(f"  {'Keyword':20s} {'SPAN':>4} {'Tot':>3}  Topics/year (count per year)")
        print(f"  {'-'*70}")

        for _, r in cat_df.iterrows():
            word = r['word']
            max_span, total, presence, best_start, topic_counts = compute_span(
                word, topic_words_by_year, years
            )

            # Visual: show topic count per year (0=dot, 1-9=number, 10+=+)
            count_str = ''
            for c in topic_counts:
                if c == 0:
                    count_str += chr(0x00B7)  # middle dot
                elif c <= 9:
                    count_str += str(c)
                else:
                    count_str += '+'

            total_topics = sum(topic_counts)

            span_rows.append({
                'subject': subject, 'word': word, 'category': cat,
                'span': max_span, 'total_years_present': total,
                'span_start': best_start if best_start else -1,
                'total_years': len(years),
                'coverage_pct': round(total / len(years) * 100, 1),
                'total_topic_hits': total_topics,
                'avg_topics_when_present': round(total_topics / total, 2) if total > 0 else 0,
                'topic_counts_per_year': str(topic_counts),
                'presence': ''.join([chr(0x2588) if p else chr(0x00B7) for p in presence]),
            })

            avg_t = f"{total_topics/total:.1f}" if total > 0 else "0"
            print(f"  {word:20s} {max_span:4d} {total:3d}  {count_str}  (avg {avg_t} topics)")

    span_df = pd.DataFrame(span_rows)
    span_df.to_csv(RESULT_DIR / subject / 'keyword_span.csv', index=False)
    print(f"\n  Year index: {'  '.join([str(y) for y in years[::5]])}")
    print(f"  Saved: {RESULT_DIR / subject / 'keyword_span.csv'}")


SPAN Analysis: CS

  📈 Emerging:
  Keyword              SPAN Tot  Topics/year (count per year)
  ----------------------------------------------------------------------
  image                  17  19  ····1·1··3++11112245756432  (avg 3.9 topics)
  dataset                13  13  ·············1111111466443  (avg 2.6 topics)
  training               12  12  ··············111112324211  (avg 1.7 topics)
  deep                   11  11  ···············11122222211  (avg 1.5 topics)
  video                  11  11  ···············12111222221  (avg 1.5 topics)
  learning               12  14  ··1·········1·111111122122  (avg 1.3 topics)
  llm                     3   3  ·······················121  (avg 1.3 topics)
  human                  11  11  ···············11111222122  (avg 1.5 topics)
  datum                  26  26  ++++++++++421221147++++++7  (avg 16.5 topics)
  deep_learning          10  10  ················1112333211  (avg 1.8 topics)
  detection              12  12  ··············111

## Step 4: Summary Analysis

In [6]:
for subject in LIST_SUBJECT:
    print(f"\n{'='*70}")
    print(f"Summary: {subject.upper()} (DTM)")
    print(f"{'='*70}")

    span_df = pd.read_csv(RESULT_DIR / subject / "keyword_span.csv")

    # Per-category summary
    summary_rows = []
    for cat in ["emerging", "stable", "decaying"]:
        cat_data = span_df[span_df["category"] == cat]
        s = {
            "subject": subject, "category": cat,
            "n_keywords": len(cat_data),
            "avg_span": round(cat_data["span"].mean(), 2),
            "max_span": int(cat_data["span"].max()),
            "min_span": int(cat_data["span"].min()),
            "avg_coverage_pct": round(cat_data["coverage_pct"].mean(), 2),
            "n_never_captured": int((cat_data["span"] == 0).sum()),
            "n_full_span": int((cat_data["span"] == cat_data["total_years"]).sum()),
        }
        summary_rows.append(s)

        icon = {"emerging": "📈", "stable": "🔒", "decaying": "📉"}[cat]
        print(f"\n  {icon} {cat.upper()}:")
        print(f"    Avg SPAN: {s['avg_span']:.1f} years (max={s['max_span']}, min={s['min_span']})")
        print(f"    Avg coverage: {s['avg_coverage_pct']:.1f}%")
        print(f"    Never captured: {s['n_never_captured']}/{s['n_keywords']}")
        print(f"    Full span (all years): {s['n_full_span']}/{s['n_keywords']}")

    summary_df = pd.DataFrame(summary_rows)
    summary_df.to_csv(RESULT_DIR / subject / "keyword_span_summary.csv", index=False)

    # Overall model score
    overall_avg_span = span_df["span"].mean()
    overall_coverage = span_df["coverage_pct"].mean()
    n_captured = (span_df["span"] > 0).sum()
    n_total = len(span_df)

    print(f"\n  ─── Overall ───")
    print(f"  Avg SPAN: {overall_avg_span:.2f} / {span_df['total_years'].iloc[0]} years")
    print(f"  Avg coverage: {overall_coverage:.1f}%")
    print(f"  Keywords captured: {n_captured}/{n_total} ({n_captured/n_total:.0%})")
    print(f"  Saved: {RESULT_DIR / subject}")


Summary: CS (DTM)

  📈 EMERGING:
    Avg SPAN: 11.8 years (max=26, min=3)
    Avg coverage: 46.4%
    Never captured: 0/20
    Full span (all years): 1/20

  🔒 STABLE:
    Avg SPAN: 17.6 years (max=26, min=10)
    Avg coverage: 78.5%
    Never captured: 0/20
    Full span (all years): 2/20

  📉 DECAYING:
    Avg SPAN: 13.6 years (max=26, min=2)
    Avg coverage: 63.1%
    Never captured: 0/20
    Full span (all years): 8/20

  ─── Overall ───
  Avg SPAN: 14.33 / 26 years
  Avg coverage: 62.6%
  Keywords captured: 60/60 (100%)
  Saved: ../../../../results/dtm/tren/cs

Summary: MATH (DTM)

  📈 EMERGING:
    Avg SPAN: 12.2 years (max=24, min=4)
    Avg coverage: 47.1%
    Never captured: 0/20
    Full span (all years): 0/20

  🔒 STABLE:
    Avg SPAN: 19.9 years (max=26, min=6)
    Avg coverage: 82.5%
    Never captured: 0/20
    Full span (all years): 8/20

  📉 DECAYING:
    Avg SPAN: 14.2 years (max=26, min=0)
    Avg coverage: 66.9%
    Never captured: 1/20
    Full span (all years): 9